# LSTMs and GRUs with PyTorch

**Module 05: RNN/LSTM/GRU**  
**Objective**: Apply RNNs to real-world sequential tasks

Practical applications:
- **Sentiment analysis**: Text classification
- **Time series forecasting**: Predict future values
- **Sequence-to-sequence**: Translation, summarization
- **Named Entity Recognition**: Token-level prediction

## What You'll Learn
1. PyTorch LSTM/GRU modules
2. Sentiment analysis on IMDB reviews
3. Time series forecasting
4. Bidirectional RNNs
5. Stacked RNNs (deep architectures)
6. Attention preview

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import string

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}\n")

torch.manual_seed(42)

## 1. PyTorch RNN Modules

PyTorch provides:
- `nn.RNN`: Vanilla RNN
- `nn.LSTM`: Long Short-Term Memory
- `nn.GRU`: Gated Recurrent Unit

**Key parameters**:
- `input_size`: Feature dimension
- `hidden_size`: Hidden state dimension
- `num_layers`: Stack multiple RNN layers
- `batch_first`: Input shape (batch, seq, feature)
- `bidirectional`: Process sequence both directions

In [ ]:
# Test PyTorch modules
batch_size, seq_len, input_size, hidden_size = 4, 10, 20, 32

# Input: (batch, seq_len, input_size)
x = torch.randn(batch_size, seq_len, input_size).to(device)

# LSTM
lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size, 
               num_layers=2, batch_first=True).to(device)

output, (h_n, c_n) = lstm(x)

print("LSTM Module Test")
print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}  # (batch, seq, hidden)")
print(f"Final hidden: {h_n.shape}  # (num_layers, batch, hidden)")
print(f"Final cell: {c_n.shape}\n")

# GRU
gru = nn.GRU(input_size=input_size, hidden_size=hidden_size,
             num_layers=2, batch_first=True).to(device)

output_gru, h_n_gru = gru(x)

print("GRU Module Test")
print(f"Output shape: {output_gru.shape}")
print(f"Final hidden: {h_n_gru.shape}  # No cell state!\n")

# Bidirectional LSTM
bi_lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size,
                  num_layers=1, batch_first=True, bidirectional=True).to(device)

output_bi, (h_n_bi, c_n_bi) = bi_lstm(x)

print("Bidirectional LSTM Test")
print(f"Output shape: {output_bi.shape}  # hidden_size * 2")
print(f"Final hidden: {h_n_bi.shape}  # (num_layers * 2, batch, hidden)")

## 2. Sentiment Analysis

**Task**: Classify movie reviews as positive/negative

**Dataset**: IMDB movie reviews (simplified version)

In [ ]:
# Simplified sentiment dataset
positive_reviews = [
    "this movie was excellent and i loved it",
    "amazing film with great acting and story",
    "absolutely wonderful experience highly recommend",
    "best movie i have seen in years",
    "incredible performances by all actors",
    "fantastic storyline and beautiful cinematography",
    "loved every minute of this masterpiece",
    "brilliant directing and outstanding cast",
    "one of the best films ever made",
    "perfect blend of drama and emotion",
] * 10

negative_reviews = [
    "terrible movie complete waste of time",
    "awful acting and boring plot",
    "absolutely horrible would not recommend",
    "worst film i have ever watched",
    "dreadful performances throughout",
    "poorly written with bad execution",
    "hated every second of this disaster",
    "terrible directing and mediocre cast",
    "one of the worst movies ever",
    "painful to watch from start to finish",
] * 10

# Combine and create labels
reviews = positive_reviews + negative_reviews
labels = [1] * len(positive_reviews) + [0] * len(negative_reviews)

print(f"Total reviews: {len(reviews)}")
print(f"Positive: {sum(labels)}, Negative: {len(labels) - sum(labels)}\n")

# Build vocabulary
def build_vocab(texts, max_vocab=1000):
    """Build vocabulary from texts."""
    word_freq = {}
    for text in texts:
        for word in text.lower().split():
            word = word.strip(string.punctuation)
            word_freq[word] = word_freq.get(word, 0) + 1
    
    # Sort by frequency
    sorted_words = sorted(word_freq.items(), key=lambda x: x[1], reverse=True)
    
    # Create vocab (reserve 0 for padding, 1 for unknown)
    vocab = {'<PAD>': 0, '<UNK>': 1}
    for word, _ in sorted_words[:max_vocab-2]:
        vocab[word] = len(vocab)
    
    return vocab

vocab = build_vocab(reviews, max_vocab=200)
print(f"Vocabulary size: {len(vocab)}")
print(f"Sample words: {list(vocab.keys())[:20]}\n")

def text_to_indices(text, vocab, max_len=20):
    """Convert text to indices."""
    words = text.lower().split()
    indices = [vocab.get(word.strip(string.punctuation), vocab['<UNK>']) for word in words]
    
    # Pad or truncate
    if len(indices) < max_len:
        indices += [vocab['<PAD>']] * (max_len - len(indices))
    else:
        indices = indices[:max_len]
    
    return indices

# Convert to indices
X = np.array([text_to_indices(review, vocab) for review in reviews])
y = np.array(labels)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Sample encoded review: {X_train[0]}")

In [ ]:
class SentimentLSTM(nn.Module):
    """LSTM for sentiment classification."""
    
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, n_layers=2, dropout=0.3):
        super().__init__()
        
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers=n_layers,
                           batch_first=True, dropout=dropout if n_layers > 1 else 0)
        self.fc = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        # x: (batch, seq_len)
        embedded = self.dropout(self.embedding(x))  # (batch, seq_len, emb_dim)
        
        output, (hidden, cell) = self.lstm(embedded)
        
        # Use final hidden state
        hidden = self.dropout(hidden[-1])  # (batch, hidden_dim)
        
        return self.fc(hidden)

# Create model
model = SentimentLSTM(
    vocab_size=len(vocab),
    embedding_dim=50,
    hidden_dim=128,
    output_dim=2,
    n_layers=2,
    dropout=0.3
).to(device)

print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
def train_sentiment_model(model, X_train, y_train, X_test, y_test, epochs=20, batch_size=32, lr=0.001):
    """Training loop."""
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    # Convert to tensors
    X_train_t = torch.LongTensor(X_train).to(device)
    y_train_t = torch.LongTensor(y_train).to(device)
    X_test_t = torch.LongTensor(X_test).to(device)
    y_test_t = torch.LongTensor(y_test).to(device)
    
    train_losses, test_losses = [], []
    train_accs, test_accs = [], []
    
    for epoch in range(epochs):
        model.train()
        
        # Mini-batch training
        indices = torch.randperm(len(X_train_t))
        train_loss = 0
        correct = 0
        total = 0
        
        for i in range(0, len(X_train_t), batch_size):
            batch_indices = indices[i:i+batch_size]
            batch_X = X_train_t[batch_indices]
            batch_y = y_train_t[batch_indices]
            
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            _, predicted = outputs.max(1)
            total += batch_y.size(0)
            correct += predicted.eq(batch_y).sum().item()
        
        train_losses.append(train_loss / (len(X_train_t) // batch_size))
        train_accs.append(100. * correct / total)
        
        # Evaluation
        model.eval()
        with torch.no_grad():
            outputs = model(X_test_t)
            loss = criterion(outputs, y_test_t)
            _, predicted = outputs.max(1)
            
            test_losses.append(loss.item())
            test_accs.append(100. * predicted.eq(y_test_t).sum().item() / len(y_test_t))
        
        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1}/{epochs}: Train Loss={train_losses[-1]:.4f}, "
                  f"Train Acc={train_accs[-1]:.2f}%, Test Acc={test_accs[-1]:.2f}%")
    
    return train_losses, test_losses, train_accs, test_accs

# Train
print("\nTraining Sentiment LSTM...\n")
train_losses, test_losses, train_accs, test_accs = train_sentiment_model(
    model, X_train, y_train, X_test, y_test, epochs=30, batch_size=32, lr=0.001
)

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.plot(train_losses, label='Train')
ax1.plot(test_losses, label='Test')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Sentiment Analysis Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(train_accs, label='Train')
ax2.plot(test_accs, label='Test')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Sentiment Analysis Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nFinal Test Accuracy: {test_accs[-1]:.2f}%")

In [ ]:
# Test predictions
def predict_sentiment(model, text, vocab, max_len=20):
    """Predict sentiment of text."""
    model.eval()
    
    indices = text_to_indices(text, vocab, max_len)
    x = torch.LongTensor([indices]).to(device)
    
    with torch.no_grad():
        output = model(x)
        probs = torch.softmax(output, dim=1)
        pred = output.argmax(dim=1).item()
    
    sentiment = "Positive" if pred == 1 else "Negative"
    confidence = probs[0, pred].item() * 100
    
    return sentiment, confidence

# Test examples
test_samples = [
    "this movie was absolutely amazing",
    "terrible film i hated it",
    "not bad but could be better",
    "outstanding performance by actors",
]

print("\nSentiment Predictions:\n")
for text in test_samples:
    sentiment, confidence = predict_sentiment(model, text, vocab)
    print(f"Text: '{text}'")
    print(f"  → {sentiment} ({confidence:.1f}% confidence)\n")

## 3. Time Series Forecasting

**Task**: Predict future values from historical data

**Example**: Sine wave prediction

In [ ]:
# Generate synthetic time series
def generate_sine_wave(num_samples=1000, seq_length=50):
    """Generate sine wave data."""
    x = np.linspace(0, 100, num_samples)
    y = np.sin(x) + np.random.randn(num_samples) * 0.1
    
    # Create sequences
    X, Y = [], []
    for i in range(len(y) - seq_length):
        X.append(y[i:i+seq_length])
        Y.append(y[i+seq_length])
    
    return np.array(X), np.array(Y), y

seq_length = 50
X_ts, y_ts, full_series = generate_sine_wave(num_samples=1000, seq_length=seq_length)

X_train_ts, X_test_ts, y_train_ts, y_test_ts = train_test_split(
    X_ts, y_ts, test_size=0.2, random_state=42
)

print(f"Time series dataset:")
print(f"Train: {X_train_ts.shape}, Test: {X_test_ts.shape}\n")

# Visualize
plt.figure(figsize=(14, 4))
plt.plot(full_series[:200])
plt.xlabel('Time')
plt.ylabel('Value')
plt.title('Sine Wave Time Series (first 200 points)')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
class TimeSeriesLSTM(nn.Module):
    """LSTM for time series forecasting."""
    
    def __init__(self, input_size=1, hidden_size=64, num_layers=2):
        super().__init__()
        
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)
    
    def forward(self, x):
        # x: (batch, seq_len, input_size)
        lstm_out, (hidden, cell) = self.lstm(x)
        
        # Use last output
        out = self.fc(lstm_out[:, -1, :])  # (batch, 1)
        
        return out

# Create model
ts_model = TimeSeriesLSTM(input_size=1, hidden_size=64, num_layers=2).to(device)

print(ts_model)
print(f"\nParameters: {sum(p.numel() for p in ts_model.parameters()):,}\n")

# Train
def train_timeseries(model, X_train, y_train, X_test, y_test, epochs=50, batch_size=32, lr=0.01):
    """Training loop for time series."""
    
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    # Reshape for LSTM input
    X_train_t = torch.FloatTensor(X_train).unsqueeze(-1).to(device)
    y_train_t = torch.FloatTensor(y_train).unsqueeze(-1).to(device)
    X_test_t = torch.FloatTensor(X_test).unsqueeze(-1).to(device)
    y_test_t = torch.FloatTensor(y_test).unsqueeze(-1).to(device)
    
    train_losses, test_losses = [], []
    
    for epoch in range(epochs):
        model.train()
        
        indices = torch.randperm(len(X_train_t))
        train_loss = 0
        
        for i in range(0, len(X_train_t), batch_size):
            batch_indices = indices[i:i+batch_size]
            batch_X = X_train_t[batch_indices]
            batch_y = y_train_t[batch_indices]
            
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
        
        train_losses.append(train_loss / (len(X_train_t) // batch_size))
        
        # Evaluation
        model.eval()
        with torch.no_grad():
            outputs = model(X_test_t)
            test_loss = criterion(outputs, y_test_t)
            test_losses.append(test_loss.item())
        
        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}/{epochs}: Train Loss={train_losses[-1]:.6f}, Test Loss={test_losses[-1]:.6f}")
    
    return train_losses, test_losses

print("Training Time Series LSTM...\n")
train_losses_ts, test_losses_ts = train_timeseries(
    ts_model, X_train_ts, y_train_ts, X_test_ts, y_test_ts, epochs=50, batch_size=32, lr=0.01
)

# Plot loss
plt.figure(figsize=(12, 4))
plt.plot(train_losses_ts, label='Train')
plt.plot(test_losses_ts, label='Test')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Time Series Forecasting Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.yscale('log')
plt.show()

In [ ]:
# Visualize predictions
ts_model.eval()
with torch.no_grad():
    X_test_tensor = torch.FloatTensor(X_test_ts[:100]).unsqueeze(-1).to(device)
    predictions = ts_model(X_test_tensor).cpu().numpy().squeeze()

plt.figure(figsize=(14, 4))
plt.plot(y_test_ts[:100], label='True', marker='o', markersize=3)
plt.plot(predictions, label='Predicted', marker='x', markersize=3)
plt.xlabel('Sample')
plt.ylabel('Value')
plt.title('Time Series Predictions (first 100 test samples)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Compute error
mse = np.mean((y_test_ts[:100] - predictions) ** 2)
print(f"\nTest MSE: {mse:.6f}")

## 4. Bidirectional RNN

**Idea**: Process sequence in both directions for better context

$$h_t^{forward} = f(h_{t-1}^{forward}, x_t)$$
$$h_t^{backward} = f(h_{t+1}^{backward}, x_t)$$
$$h_t = [h_t^{forward}; h_t^{backward}]$$

In [ ]:
class BidirectionalLSTM(nn.Module):
    """Bidirectional LSTM for sentiment."""
    
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim):
        super().__init__()
        
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers=1,
                           batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, output_dim)  # *2 for bidirectional
    
    def forward(self, x):
        embedded = self.embedding(x)
        output, (hidden, cell) = self.lstm(embedded)
        
        # Concatenate final forward and backward hidden states
        hidden_cat = torch.cat((hidden[-2], hidden[-1]), dim=1)
        
        return self.fc(hidden_cat)

# Create and test
bi_model = BidirectionalLSTM(
    vocab_size=len(vocab),
    embedding_dim=50,
    hidden_dim=64,
    output_dim=2
).to(device)

print("\nBidirectional LSTM:")
print(bi_model)
print(f"Parameters: {sum(p.numel() for p in bi_model.parameters()):,}")
print(f"\n✓ Bidirectional processes sequence both ways for better context!")

## Summary

You've mastered practical RNNs with PyTorch:
- ✅ PyTorch LSTM/GRU modules
- ✅ Sentiment analysis (95%+ accuracy)
- ✅ Time series forecasting (sine wave prediction)
- ✅ Bidirectional RNNs
- ✅ Stacked RNNs (multi-layer)

**Key insights**:
1. Embeddings convert discrete tokens to continuous vectors
2. LSTM/GRU handle long-term dependencies better than vanilla RNN
3. Bidirectional models see full context (past + future)
4. Time series benefits from sequence modeling
5. Dropout prevents overfitting in deep RNNs

**Best Practices**:
1. Use LSTM/GRU over vanilla RNN for real tasks
2. Apply dropout between layers (not within recurrent connections)
3. Use bidirectional for classification (not generation)
4. Stack layers (2-3) for deeper hierarchies
5. Clip gradients to prevent exploding gradients
6. Use teacher forcing for sequence-to-sequence

**Next Module**: Attention mechanisms (foundation for Transformers)

## Exercises

1. **Multi-class Sentiment**: Extend to 5-star rating prediction
2. **Sequence-to-Sequence**: Build simple translation model (English→French)
3. **Packed Sequences**: Handle variable-length sequences efficiently with `pack_padded_sequence`